# 31 Entity Disambiguation

Phase 31 execution notebook for layered entity disambiguation restart.

The notebook orchestrates the pipeline in separate phase cells so each stage can be debugged independently.

This notebook performs one deterministic automated run and writes aligned artifacts for manual downstream reconciliation.

In [ ]:
from pathlib import Path
from datetime import UTC, datetime
import os
import sys

import pandas as pd

repo_root = Path.cwd()
if not (repo_root / "speakermining").exists():
    # Notebook may be opened from the notebooks folder.
    for parent in [repo_root] + list(repo_root.parents):
        if (parent / "speakermining").exists():
            repo_root = parent
            break

os.chdir(repo_root)
src_path = repo_root / "speakermining" / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))


from process.entity_disambiguation.contracts import INPUT_FILES
from process.entity_disambiguation.orchestrator import build_source_inventory_report
from process.io_guardrails import atomic_write_csv


CLASS_SOURCE_PLANS: dict[str, list[tuple[str, str | None]]] = {
    "broadcasting_programs": [
        ("setup", "setup_broadcasting_programs"),
        ("wikidata", "wikidata_programs"),
        ("fernsehserien_de", None),
    ],
    "seasons": [
        ("zdf_archive", "zdf_seasons"),
        ("wikidata", "wikidata_series"),
        ("fernsehserien_de", None),
    ],
    "episodes": [
        ("zdf_archive", "zdf_episodes"),
        ("wikidata", "wikidata_episodes"),
        ("fernsehserien_de", "fs_episode_metadata"),
    ],
    "persons": [
        ("zdf_archive", "zdf_persons"),
        ("wikidata", "wikidata_persons"),
        ("fernsehserien_de", "fs_episode_guests"),
    ],
    "topics": [
        ("zdf_archive", "zdf_topics"),
        ("wikidata", "wikidata_topics"),
        ("fernsehserien_de", None),
    ],
    "roles": [
        ("zdf_archive", "zdf_persons"),
        ("wikidata", "wikidata_roles"),
        ("fernsehserien_de", None),
    ],
    "organizations": [
        ("zdf_archive", None),
        ("wikidata", "wikidata_organizations"),
        ("fernsehserien_de", None),
    ],
}


def write_example(df: pd.DataFrame, out_path: Path) -> None:
    from process.io_guardrails import atomic_write_csv

    out_path.parent.mkdir(parents=True, exist_ok=True)
    atomic_write_csv(out_path, df.head(1), index=False)


def _class_inventory_rows(class_name: str) -> pd.DataFrame:
    rows: list[dict[str, str | int]] = []
    plan = CLASS_SOURCE_PLANS.get(class_name, [])
    for source_label, source_key in plan:
        if source_key is None:
            rows.append({"source": source_label, "instances": "n.a.", "properties": "n.a."})
            continue

        if source_key not in source_inventory.index:
            rows.append({"source": source_label, "instances": "n.a.", "properties": "n.a."})
            continue

        rows.append(
            {
                "source": source_label,
                "instances": int(source_inventory.loc[source_key, "instances"]),
                "properties": int(source_inventory.loc[source_key, "properties"]),
            }
        )

    return pd.DataFrame(rows)


def print_alignment_progress(class_name: str, frame: pd.DataFrame) -> pd.DataFrame:
    class_inventory = _class_inventory_rows(class_name)
    unresolved = int(frame["match_tier"].eq("unresolved").sum()) if "match_tier" in frame.columns else 0
    print(class_name)
    print(class_inventory.to_string(index=False))
    print(f"aligned rows: {int(len(frame))}; unresolved: {unresolved}")
    return class_inventory


run_id = datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")
repo_root

In [ ]:
from process.entity_disambiguation.io_staging import stage_inputs

staging_manifest = stage_inputs()
staging_manifest

In [ ]:
source_inventory = build_source_inventory_report().set_index("source_name")
source_inventory

## Phase 2: Value Normalization
Normalize source values before matching so later phases can reason over stable text, dates, and identifiers.

In [ ]:
from process.entity_disambiguation.normalization import normalize_inputs

normalized = normalize_inputs()
{key: df.shape for key, df in normalized.items()}

## Phase 3: Schema Harmonization
Map source properties to canonical columns and persist the schema contract for later alignment.

In [ ]:
from process.entity_disambiguation.schema_mapping import write_schema_mapping

schema_mapping = write_schema_mapping(normalized)
schema_mapping

## Phase 4: Layered Alignment and Persistence
Build the aligned tables, write the artifacts, and emit one-record examples for each written file.

In [ ]:
import json

from pathlib import Path

from process.io_guardrails import atomic_write_csv, atomic_write_text
from process.entity_disambiguation.contracts import LAYERED_EXAMPLES_DIR, NORMALIZED_EXAMPLES_DIR, OUTPUT_FILES, RAW_EXAMPLES_DIR, SCHEMA_EXAMPLES_DIR
from process.entity_disambiguation.evidence import combine_evidence_rows
from process.entity_disambiguation.episode_alignment import build_aligned_episodes
from process.entity_disambiguation.orchestrator import _build_aligned_broadcasting_programs, _build_aligned_seasons
from process.entity_disambiguation.person_alignment import build_aligned_persons
from process.entity_disambiguation.role_org_alignment import build_aligned_organizations, build_aligned_roles
from process.entity_disambiguation.topic_alignment import build_aligned_topics

In [ ]:
aligned_broadcasting_programs = _build_aligned_broadcasting_programs(normalized)
atomic_write_csv(OUTPUT_FILES["aligned_broadcasting_programs"], aligned_broadcasting_programs, index=False)
write_example(aligned_broadcasting_programs, LAYERED_EXAMPLES_DIR / "aligned_broadcasting_programs.example.csv")

broadcasting_program_inventory = print_alignment_progress("broadcasting_programs", aligned_broadcasting_programs)
broadcasting_program_evidence = [
    {
        "alignment_unit_id": str(row["alignment_unit_id"]),
        "entity_class": "broadcasting_program",
        "match_strategy": str(row["match_strategy"]),
        "match_tier": str(row["match_tier"]),
        "match_confidence": float(row["match_confidence"]),
        "evidence_summary": str(row["evidence_summary"]),
        "unresolved_reason_code": str(row["unresolved_reason_code"]),
    }
    for _, row in aligned_broadcasting_programs.iterrows()
]
len(aligned_broadcasting_programs)
aligned_broadcasting_programs

In [ ]:
aligned_seasons = _build_aligned_seasons(normalized)
atomic_write_csv(OUTPUT_FILES["aligned_seasons"], aligned_seasons, index=False)
write_example(aligned_seasons, LAYERED_EXAMPLES_DIR / "aligned_seasons.example.csv")

seasons_inventory = print_alignment_progress("seasons", aligned_seasons)
season_evidence = []
for _, row in aligned_seasons.iterrows():
    season_evidence.append(
        {
            "alignment_unit_id": str(row.get("alignment_unit_id", "")),
            "entity_class": "season",
            "match_strategy": str(row.get("match_strategy", "")),
            "match_tier": str(row.get("match_tier", "")),
            "match_confidence": float(row.get("match_confidence", 0.0) or 0.0),
            "evidence_summary": str(row.get("evidence_summary", "")),
            "unresolved_reason_code": str(row.get("unresolved_reason_code", "")),
        }
    )
len(aligned_seasons)
aligned_seasons

In [ ]:
aligned_episodes, episode_evidence = build_aligned_episodes(normalized)
atomic_write_csv(OUTPUT_FILES["aligned_episodes"], aligned_episodes, index=False)
write_example(aligned_episodes, LAYERED_EXAMPLES_DIR / "aligned_episodes.example.csv")
episodes_inventory = print_alignment_progress("episodes", aligned_episodes)
aligned_episodes

In [ ]:
aligned_persons, person_evidence = build_aligned_persons(normalized, aligned_episodes)
atomic_write_csv(OUTPUT_FILES["aligned_persons"], aligned_persons, index=False)
write_example(aligned_persons, LAYERED_EXAMPLES_DIR / "aligned_persons.example.csv")
persons_inventory = print_alignment_progress("persons", aligned_persons)
aligned_persons

In [ ]:
aligned_topics, topic_evidence = build_aligned_topics(normalized)
atomic_write_csv(OUTPUT_FILES["aligned_topics"], aligned_topics, index=False)
write_example(aligned_topics, LAYERED_EXAMPLES_DIR / "aligned_topics.example.csv")
topics_inventory = print_alignment_progress("topics", aligned_topics)
aligned_topics

In [ ]:
aligned_roles, role_evidence = build_aligned_roles(normalized)
atomic_write_csv(OUTPUT_FILES["aligned_roles"], aligned_roles, index=False)
write_example(aligned_roles, LAYERED_EXAMPLES_DIR / "aligned_roles.example.csv")
roles_inventory = print_alignment_progress("roles", aligned_roles)
aligned_roles

In [ ]:
aligned_organizations, organization_evidence = build_aligned_organizations(normalized)
atomic_write_csv(OUTPUT_FILES["aligned_organizations"], aligned_organizations, index=False)
write_example(aligned_organizations, LAYERED_EXAMPLES_DIR / "aligned_organizations.example.csv")
organizations_inventory = print_alignment_progress("organizations", aligned_organizations)
aligned_organizations

In [ ]:
evidence_df = combine_evidence_rows(
    broadcasting_program_evidence,
    season_evidence,
    episode_evidence,
    person_evidence,
    topic_evidence,
    role_evidence,
    organization_evidence,
)
atomic_write_csv(OUTPUT_FILES["match_evidence"], evidence_df, index=False)

write_example(staging_manifest, RAW_EXAMPLES_DIR / "staging_manifest.example.csv")
for _, manifest_row in staging_manifest.iterrows():
    staged_path = Path(str(manifest_row.get("staged_path", "")))
    if not staged_path.exists():
        continue

    suffix = staged_path.suffix.lower()
    example_name = f"{staged_path.stem}.example{suffix}"
    example_path = RAW_EXAMPLES_DIR / example_name

    if suffix == ".csv":
        sample_df = pd.read_csv(staged_path, nrows=1, dtype=str).fillna("")
        atomic_write_csv(example_path, sample_df, index=False)
    elif suffix == ".json":
        with staged_path.open("r", encoding="utf-8") as f:
            payload = json.load(f)
        if isinstance(payload, dict):
            if payload:
                first_key = sorted(payload.keys())[0]
                sample_payload = {first_key: payload[first_key]}
            else:
                sample_payload = {}
        elif isinstance(payload, list):
            sample_payload = payload[:1]
        else:
            sample_payload = payload
        atomic_write_text(example_path, json.dumps(sample_payload, indent=2, ensure_ascii=True) + "\n")

for key, df in normalized.items():
    write_example(df, NORMALIZED_EXAMPLES_DIR / f"{key}.example.csv")
write_example(schema_mapping, SCHEMA_EXAMPLES_DIR / "source_schema_mapping.example.csv")

{
    "broadcasting_programs": len(aligned_broadcasting_programs),
    "seasons": len(aligned_seasons),
    "episodes": len(aligned_episodes),
    "persons": len(aligned_persons),
    "topics": len(aligned_topics),
    "roles": len(aligned_roles),
    "organizations": len(aligned_organizations),
    "evidence_rows": len(evidence_df),
}

## Phase 5: Quality Gates and Run Summary
Validate the written artifacts and persist the run summary for downstream review.

In [ ]:
import json

from process.entity_disambiguation.contracts import OUTPUT_FILES
from process.entity_disambiguation.quality_gates import run_quality_gates
from process.io_guardrails import atomic_write_text

gate_report = run_quality_gates()
summary = {
    "run_id": run_id,
    "write_examples": True,
    "staged_files": int(len(staging_manifest)),
    "source_inventory": source_inventory.reset_index().to_dict(orient="records"),
    "class_source_inventory": {
        "broadcasting_programs": broadcasting_program_inventory.to_dict(orient="records"),
        "seasons": seasons_inventory.to_dict(orient="records"),
        "episodes": episodes_inventory.to_dict(orient="records"),
        "persons": persons_inventory.to_dict(orient="records"),
        "topics": topics_inventory.to_dict(orient="records"),
        "roles": roles_inventory.to_dict(orient="records"),
        "organizations": organizations_inventory.to_dict(orient="records"),
    },
    "aligned_counts": {
        "broadcasting_programs": int(len(aligned_broadcasting_programs)),
        "seasons": int(len(aligned_seasons)),
        "episodes": int(len(aligned_episodes)),
        "persons": int(len(aligned_persons)),
        "topics": int(len(aligned_topics)),
        "roles": int(len(aligned_roles)),
        "organizations": int(len(aligned_organizations)),
    },
    "unmatched_counts": {
        "broadcasting_programs": int(aligned_broadcasting_programs["match_tier"].eq("unresolved").sum()),
        "seasons": int(aligned_seasons["match_tier"].eq("unresolved").sum()),
        "episodes": int(aligned_episodes["match_tier"].eq("unresolved").sum()),
        "persons": int(aligned_persons["match_tier"].eq("unresolved").sum()),
        "topics": int(aligned_topics["match_tier"].eq("unresolved").sum()),
        "roles": int(aligned_roles["match_tier"].eq("unresolved").sum()),
        "organizations": int(aligned_organizations["match_tier"].eq("unresolved").sum()),
    },
    "quality_gates": gate_report,
}
atomic_write_text(OUTPUT_FILES["run_summary"], json.dumps(summary, indent=2, ensure_ascii=True) + "\n")
summary